In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import scanpy.external as sce
import scipy
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import pandas as pd

In [ ]:
path = "marker.csv"
cts = ['GCBC','NBC_MBC','FDC','epithelial','CD4_T','myeloid'] 
df = pd.read_csv(path)
df

In [ ]:
cts = ['GCBC','NBC_MBC','FDC','epithelial','CD4_T','myeloid'] 
n_genes = [1000, 1000, 1000, 1000, 1000, 1000]

In [ ]:
def process(adj):
    np.fill_diagonal(adj, 0)
    adj = np.where(adj > 0, adj, 0)
    adj = adj / adj.max()
    return adj

In [ ]:
def cosine_similarity(X, device='cuda'):
    X = torch.tensor(X, dtype=torch.float32)
    X = X.T
    X = torch.tensor(X)
    X_normalized = F.normalize(X, p=2, dim=1)
    cos = X_normalized @ X_normalized.T 
    cos= cos.numpy()
    return cos
def topk(sim_matrix, k):
    N = sim_matrix.shape[0]
    adj = np.zeros_like(sim_matrix)

    for i in range(N):
        row = sim_matrix[i].copy()
        row[i] = -np.inf 
        topk_idx = np.argpartition(-row, k)[:k]
        adj[i, topk_idx] = sim_matrix[i, topk_idx]

    adj = np.maximum(adj, adj.T)
    return adj

In [ ]:
adata = sc.read_h5ad('data/spatial/processed_data/scRNA.h5ad')
for ct in cts:
    genes_ct = df[ct].dropna().to_list()
    adata_ct = adata[adata.obs['cell_type']==ct].copy()
    adata_ct = adata_ct[:, genes_ct].copy()
    adj_ct = process(cosine_similarity(adata_ct.X.toarray()))
    #threshold = np.percentile(adj_ct, 90)


    #adj_ct = np.where(adj_ct >= threshold, adj_ct, 0)
    adj_ct = topk(adj_ct, 100)

    degree = np.sum(adj_ct, axis=1) 
    isolated_nodes = np.where(degree == 0)[0]
    if len(isolated_nodes) > 0:
        print(f"Graph has isolated nodes: {isolated_nodes}")
    else:
        print("No isolated nodes in the graph.")

    torch.save(adj_ct,f'data/spatial/graph/intra/{ct}.pt')